<a href="https://colab.research.google.com/github/jluisro-byte/Agent-Skills-for-Context-Engineering/blob/main/Resistor_vlm_proto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM GB:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2,
        ),
    )

Torch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
VRAM GB: 14.56


In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
%cd /content
!git clone -b feature/vlm-pipeline \
  https://github.com/jluisro-byte/Resistor-VLM.git

%cd /content/Resistor-VLM

/content
fatal: destination path 'Resistor-VLM' already exists and is not an empty directory.
/content/Resistor-VLM


In [4]:
from google.colab import userdata
import requests
import zipfile
import io
import os
import shutil

token = userdata.get("GITHUB_TOKEN")

if not token:
    raise RuntimeError(
        "No se encontró el secreto GITHUB_TOKEN en Colab."
    )

owner = "jluisro-byte"
repository = "Resistor-VLM"
branch = "feature/vlm-pipeline"

url = (
    f"https://api.github.com/repos/"
    f"{owner}/{repository}/zipball/{branch}"
)

response = requests.get(
    url,
    headers={
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
    },
    timeout=120,
)

response.raise_for_status()

destination = "/content/Resistor-VLM"

if os.path.exists(destination):
    shutil.rmtree(destination)

with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    archive.extractall("/content/github_download")

extracted = [
    os.path.join("/content/github_download", name)
    for name in os.listdir("/content/github_download")
]

if len(extracted) != 1:
    raise RuntimeError(
        f"Contenido inesperado al descomprimir: {extracted}"
    )

shutil.move(extracted[0], destination)
shutil.rmtree("/content/github_download")

print("Proyecto descargado en:", destination)

Proyecto descargado en: /content/Resistor-VLM


In [5]:
%cd /content/Resistor-VLM


/content/Resistor-VLM


In [8]:
!pwd
!ls

/content/Resistor-VLM
Resistor-VLM


In [9]:
%cd /content/Resistor-VLM
!git pull

/content/Resistor-VLM
fatal: not a git repository (or any of the parent directories): .git


In [10]:
%cd /content
!git clone -b feature/vlm-pipeline \
  https://github.com/jluisro-byte/Resistor-VLM.git

%cd /content/Resistor-VLM


/content
fatal: destination path 'Resistor-VLM' already exists and is not an empty directory.
/content/Resistor-VLM


In [11]:
!python -m pip install --upgrade pip
!python -m pip install -e ".[vlm-qwen]"
!python -m pip install qwen-vl-utils accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.0 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Obtaining file:///content/Resistor-VLM
ERROR: file:///content/Resistor-VLM does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 106.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [qwen-vl-utils]


In [12]:
import ultralytics
import transformers
import qwen_vl_utils
import resistor_vlm

print("Ultralytics:", ultralytics.__version__)
print("Transformers:", transformers.__version__)
print("Resistor-VLM importado correctamente")

ModuleNotFoundError: No module named 'ultralytics'

In [13]:
!mkdir -p models/detector
!mkdir -p data/sample_images/resistors
!mkdir -p outputs/prototype

!cp "/content/drive/MyDrive/Resistor-VLM-assets/detector/best.pt" \
    "models/detector/best.pt"

!cp "/content/drive/MyDrive/Resistor-VLM-assets/images/resistor001.jpg" \
    "data/sample_images/resistors/resistor001.jpg"

In [2]:
from pathlib import Path

paths = [
    Path("models/detector/best.pt"),
    Path("data/sample_images/resistors/resistor001.jpg"),
]

for path in paths:
    print(path, path.exists())

models/detector/best.pt False
data/sample_images/resistors/resistor001.jpg False


In [15]:
!python scripts/run_pipeline.py \
  --image "data/sample_images/resistors/resistor001.jpg" \
  --detector-model "models/detector/best.pt" \
  --vlm-model "Qwen/Qwen2.5-VL-3B-Instruct" \
  --device "cuda" \
  --output-dir "outputs/prototype" \
  --config "configs/default.yaml" \
  --verbose

python3: can't open file '/content/Resistor-VLM/scripts/run_pipeline.py': [Errno 2] No such file or directory


In [16]:
import shutil
from pathlib import Path

project_dir = Path("/content/Resistor-VLM")

if project_dir.exists():
    shutil.rmtree(project_dir)

print("Carpeta anterior eliminada")

Carpeta anterior eliminada


In [17]:
from google.colab import userdata
from pathlib import Path
import io
import requests
import shutil
import zipfile

token = userdata.get("GITHUB_TOKEN")

if not token:
    raise RuntimeError(
        "No se encontró GITHUB_TOKEN. Agrégalo en Secrets y habilita su acceso."
    )

owner = "jluisro-byte"
repository = "Resistor-VLM"
branch = "feature/vlm-pipeline"

url = (
    f"https://api.github.com/repos/"
    f"{owner}/{repository}/zipball/{branch}"
)

response = requests.get(
    url,
    headers={
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    },
    timeout=180,
)

print("GitHub HTTP status:", response.status_code)
response.raise_for_status()

download_dir = Path("/content/github_download")
project_dir = Path("/content/Resistor-VLM")

shutil.rmtree(download_dir, ignore_errors=True)
shutil.rmtree(project_dir, ignore_errors=True)
download_dir.mkdir(parents=True)

with zipfile.ZipFile(io.BytesIO(response.content)) as archive:
    archive.extractall(download_dir)

folders = [item for item in download_dir.iterdir() if item.is_dir()]

if len(folders) != 1:
    raise RuntimeError(f"Contenido descargado inesperado: {folders}")

shutil.move(str(folders[0]), str(project_dir))
shutil.rmtree(download_dir)

print("Proyecto descargado en:", project_dir)

GitHub HTTP status: 200
Proyecto descargado en: /content/Resistor-VLM


In [18]:
from pathlib import Path

project = Path("/content/Resistor-VLM")

required = [
    project / "pyproject.toml",
    project / "resistor_vlm",
    project / "scripts" / "run_pipeline.py",
    project / "configs" / "default.yaml",
]

for path in required:
    print(path.relative_to(project), "→", path.exists())

pyproject.toml → False
resistor_vlm → False
scripts/run_pipeline.py → False
configs/default.yaml → False


In [3]:
from google.colab import userdata

try:
    token = userdata.get("GITHUB_TOKEN")
    print("Token encontrado:", bool(token))
except Exception as e:
    print("Error:", e)

Token encontrado: True


In [21]:
https://github.com/jluisro-byte/Resistor-VLM


SyntaxError: invalid syntax (3752214356.py, line 1)

In [22]:
git branch -a

SyntaxError: invalid syntax (1256456309.py, line 1)

In [23]:
from pathlib import Path
import shutil

for path in [
    Path("/content/Resistor-VLM"),
    Path("/content/github_download"),
    Path("/content/resistor-vlm.zip"),
]:
    if path.is_dir():
        shutil.rmtree(path, ignore_errors=True)
    elif path.exists():
        path.unlink()

print("Espacio de trabajo limpio")

Espacio de trabajo limpio


In [24]:
from google.colab import userdata
import requests

token = userdata.get("GITHUB_TOKEN")

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

repo_url = "https://api.github.com/repos/jluisro-byte/Resistor-VLM"

response = requests.get(
    repo_url,
    headers=headers,
    timeout=60,
)

print("HTTP:", response.status_code)

if response.status_code != 200:
    print(response.text[:1000])
    raise RuntimeError(
        "El token no puede acceder al repositorio. "
        "Revisa permisos y acceso al repositorio."
    )

repo_info = response.json()

print("Repositorio:", repo_info["full_name"])
print("Privado:", repo_info["private"])
print("Rama predeterminada:", repo_info["default_branch"])

HTTP: 200
Repositorio: jluisro-byte/Resistor-vlm
Privado: True
Rama predeterminada: main


In [25]:
branch_url = (
    "https://api.github.com/repos/"
    "jluisro-byte/Resistor-VLM/"
    "branches/feature/vlm-pipeline"
)

response = requests.get(
    branch_url,
    headers=headers,
    timeout=60,
)

print("HTTP:", response.status_code)

if response.status_code != 200:
    print(response.text[:1000])
    raise RuntimeError(
        "GitHub no permitió acceder a la rama feature/vlm-pipeline."
    )

branch_info = response.json()

print("Rama:", branch_info["name"])
print("Commit:", branch_info["commit"]["sha"])

HTTP: 200
Rama: feature/vlm-pipeline
Commit: 8789033da1388d086a792248faad882f4914fe97


In [26]:
from pathlib import Path
import requests

archive_url = (
    "https://api.github.com/repos/"
    "jluisro-byte/Resistor-VLM/"
    "zipball/feature/vlm-pipeline"
)

zip_path = Path("/content/resistor-vlm.zip")

response = requests.get(
    archive_url,
    headers=headers,
    timeout=300,
    allow_redirects=True,
)

print("HTTP:", response.status_code)
print("Tipo:", response.headers.get("content-type"))
print("Tamaño descargado:", len(response.content), "bytes")

if response.status_code != 200:
    print(response.text[:1000])
    raise RuntimeError("No se pudo descargar el archivo del repositorio.")

if len(response.content) < 10_000:
    print(response.text[:1000])
    raise RuntimeError(
        "La respuesta es demasiado pequeña y probablemente no es el ZIP."
    )

zip_path.write_bytes(response.content)

print("ZIP guardado en:", zip_path)
print("Tamaño:", zip_path.stat().st_size)

HTTP: 200
Tipo: application/zip
Tamaño descargado: 230309 bytes
ZIP guardado en: /content/resistor-vlm.zip
Tamaño: 230309


In [27]:
from pathlib import Path
import shutil
import zipfile

zip_path = Path("/content/resistor-vlm.zip")
extract_root = Path("/content/github_download")
project_dir = Path("/content/Resistor-VLM")

shutil.rmtree(extract_root, ignore_errors=True)
shutil.rmtree(project_dir, ignore_errors=True)

extract_root.mkdir(parents=True)

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(extract_root)

folders = [
    item
    for item in extract_root.iterdir()
    if item.is_dir()
]

print("Carpetas extraídas:", folders)

if len(folders) != 1:
    raise RuntimeError(
        f"Se esperaba una carpeta raíz y se encontraron: {folders}"
    )

shutil.move(str(folders[0]), str(project_dir))
shutil.rmtree(extract_root)

print("Proyecto instalado en:", project_dir)

Carpetas extraídas: [PosixPath('/content/github_download/jluisro-byte-Resistor-vlm-8789033da1388d086a792248faad882f4914fe97')]
Proyecto instalado en: /content/Resistor-VLM


In [28]:
from pathlib import Path

project = Path("/content/Resistor-VLM")

required = [
    project / "pyproject.toml",
    project / "resistor_vlm" / "__init__.py",
    project / "scripts" / "run_pipeline.py",
    project / "configs" / "default.yaml",
]

all_ok = True

for path in required:
    exists = path.exists()
    print(path.relative_to(project), "→", exists)
    all_ok = all_ok and exists

if not all_ok:
    raise RuntimeError(
        "La rama descargada no contiene todos los archivos requeridos."
    )

print("Repositorio descargado correctamente")

pyproject.toml → False
resistor_vlm/__init__.py → False
scripts/run_pipeline.py → False
configs/default.yaml → False


RuntimeError: La rama descargada no contiene todos los archivos requeridos.

In [29]:
from pathlib import Path

content = Path("/content")

matches = list(content.rglob("pyproject.toml"))

print("Coincidencias encontradas:")

for path in matches:
    print(path)

KeyboardInterrupt: 

In [30]:
from pathlib import Path
import requests

archive_url = (
    "https://api.github.com/repos/"
    "jluisro-byte/Resistor-VLM/"
    "zipball/feature/vlm-pipeline"
)

zip_path = Path("/content/resistor-vlm.zip")

response = requests.get(
    archive_url,
    headers=headers,
    timeout=300,
    allow_redirects=True,
)

print("HTTP:", response.status_code)
print("Tipo:", response.headers.get("content-type"))
print("Tamaño descargado:", len(response.content), "bytes")

if response.status_code != 200:
    print(response.text[:1000])
    raise RuntimeError("No se pudo descargar el archivo del repositorio.")

if len(response.content) < 10_000:
    print(response.text[:1000])
    raise RuntimeError(
        "La respuesta es demasiado pequeña y probablemente no es el ZIP."
    )

zip_path.write_bytes(response.content)

print("ZIP guardado en:", zip_path)
print("Tamaño:", zip_path.stat().st_size)

HTTP: 200
Tipo: application/zip
Tamaño descargado: 230309 bytes
ZIP guardado en: /content/resistor-vlm.zip
Tamaño: 230309


In [31]:
from pathlib import Path
import shutil
import zipfile

zip_path = Path("/content/resistor-vlm.zip")
extract_root = Path("/content/github_download")
project_dir = Path("/content/Resistor-VLM")

shutil.rmtree(extract_root, ignore_errors=True)
shutil.rmtree(project_dir, ignore_errors=True)

extract_root.mkdir(parents=True)

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(extract_root)

folders = [
    item
    for item in extract_root.iterdir()
    if item.is_dir()
]

print("Carpetas extraídas:", folders)

if len(folders) != 1:
    raise RuntimeError(
        f"Se esperaba una carpeta raíz y se encontraron: {folders}"
    )

shutil.move(str(folders[0]), str(project_dir))
shutil.rmtree(extract_root)

print("Proyecto instalado en:", project_dir)

Carpetas extraídas: [PosixPath('/content/github_download/jluisro-byte-Resistor-vlm-8789033da1388d086a792248faad882f4914fe97')]
Proyecto instalado en: /content/Resistor-VLM


In [32]:
from pathlib import Path

project = Path("/content/Resistor-VLM")

required = [
    project / "pyproject.toml",
    project / "resistor_vlm" / "__init__.py",
    project / "scripts" / "run_pipeline.py",
    project / "configs" / "default.yaml",
]

all_ok = True

for path in required:
    exists = path.exists()
    print(path.relative_to(project), "→", exists)
    all_ok = all_ok and exists

if not all_ok:
    raise RuntimeError(
        "La rama descargada no contiene todos los archivos requeridos."
    )

print("Repositorio descargado correctamente")

pyproject.toml → False
resistor_vlm/__init__.py → False
scripts/run_pipeline.py → False
configs/default.yaml → False


RuntimeError: La rama descargada no contiene todos los archivos requeridos.

In [7]:
from pathlib import Path

print("=== Contenido de /content ===")
for p in Path("/content").iterdir():
    print(p)

=== Contenido de /content ===
/content/.config
/content/drive
/content/sample_data


In [6]:
from pathlib import Path

print("=== Buscando pyproject.toml ===")

for p in Path("/content").rglob("pyproject.toml"):
    print(p)

=== Buscando pyproject.toml ===


KeyboardInterrupt: 

In [35]:
from pathlib import Path

print("=== Buscando run_pipeline.py ===")

for p in Path("/content").rglob("run_pipeline.py"):
    print(p)

=== Buscando run_pipeline.py ===
/content/Resistor-VLM/Resistor-VLM/scripts/run_pipeline.py


In [5]:
%cd /content/Resistor-VLM/Resistor-VLM


[Errno 2] No such file or directory: '/content/Resistor-VLM/Resistor-VLM'
/content


In [4]:
from pathlib import Path

print(Path("pyproject.toml").exists())
print(Path("scripts/run_pipeline.py").exists())
print(Path("resistor_vlm").exists())
print(Path("configs/default.yaml").exists())

False
False
False
False


In [38]:
!python -m pip install --upgrade pip
!python -m pip install -e ".[vlm-qwen]"
!python -m pip install ultralytics qwen-vl-utils accelerate

Obtaining file:///content/Resistor-VLM/Resistor-VLM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for resistor-vlm (pyproject.toml) ... done
  Created wheel for resistor-vlm: filename=resistor_vlm-1.2.1-0.editable-py3-none-any.whl size=5607 sha256=b1f657e62043d48af233731bbc0295db497311f64ecfe118fc36b7e6e706c0a3
  Stored in directory: /tmp/pip-ephem-wheel-cache-4wop99nh/wheels/d4/ed/9a/0289066609953a1ea68f8becbe5698a35d3069018739987054
Successfully built resistor-vlm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ultralytics]


In [39]:
import ultralytics
import transformers
import qwen_vl_utils
import resistor_vlm

print("Ultralytics:", ultralytics.__version__)
print("Transformers:", transformers.__version__)
print("Qwen utils OK")
print("Resistor-VLM OK")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics: 8.4.115
Transformers: 5.13.1
Qwen utils OK
Resistor-VLM OK


In [40]:
!mkdir -p models/detector
!mkdir -p data/sample_images/resistors

!cp "/content/drive/MyDrive/Resistor-VLM-assets/detector/best.pt" \
      models/detector/

!cp "/content/drive/MyDrive/Resistor-VLM-assets/images/resistor001.jpg" \
      data/sample_images/resistors/

In [41]:
!python scripts/run_pipeline.py \
    --image data/sample_images/resistors/resistor001.jpg \
    --detector-model models/detector/best.pt \
    --vlm-model Qwen/Qwen2.5-VL-3B-Instruct \
    --device cuda \
    --output-dir outputs/prototype \
    --config configs/default.yaml \
    --verbose

INFO:resistor_vlm.config.loader:Loaded configuration from configs/default.yaml
DEBUG:pydot:pydot initializing
DEBUG:pydot:pydot 4.0.1
DEBUG:pydot.core:pydot core module initializing
DEBUG:matplotlib:matplotlib data path: /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data
DEBUG:matplotlib:CONFIGDIR=/root/.config/matplotlib
DEBUG:matplotlib:interactive is False
DEBUG:matplotlib:platform is linux
DEBUG:matplotlib:CACHEDIR=/root/.cache/matplotlib
DEBUG:matplotlib.font_manager:Using fontManager instance from /root/.cache/matplotlib/fontlist-v390.json
INFO:resistor_vlm.detector.backends.model_loader:Loaded model best.pt into cache
INFO:resistor_vlm.detector.backends.yolo_backend:YOLO backend loaded model best.pt on cuda
INFO:resistor_vlm.detector.global_detector:Global detection completed for data/sample_images/resistors/resistor001.jpg: 0 detection(s) in 5954.705 ms
DEBUG:resistor_vlm.detector.tiled.tile_detector:Tile r000_c000 at (0, 0): 0 raw, 0 transformed
DEBUG:resistor_vlm.det

In [42]:
from huggingface_hub import snapshot_download

qwen_path = snapshot_download(
    repo_id="Qwen/Qwen2.5-VL-3B-Instruct",
    local_dir="/content/models/Qwen2.5-VL-3B-Instruct",
    local_files_only=False,
)

print("Modelo descargado en:", qwen_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Modelo descargado en: /content/models/Qwen2.5-VL-3B-Instruct


In [43]:
from pathlib import Path

model_dir = Path("/content/models/Qwen2.5-VL-3B-Instruct")

required = [
    model_dir / "config.json",
    model_dir / "model.safetensors.index.json",
    model_dir / "tokenizer.json",
    model_dir / "preprocessor_config.json",
]

for path in required:
    print(path.name, "→", path.exists())

weights = list(model_dir.glob("*.safetensors"))
print("Archivos safetensors:", len(weights))
for weight in weights:
    print(weight.name, round(weight.stat().st_size / 1024**3, 2), "GB")

config.json → True
model.safetensors.index.json → True
tokenizer.json → True
preprocessor_config.json → True
Archivos safetensors: 2
model-00001-of-00002.safetensors 3.71 GB
model-00002-of-00002.safetensors 3.28 GB


In [44]:
import torch
from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
)

model_path = "/content/models/Qwen2.5-VL-3B-Instruct"

print("GPU:", torch.cuda.get_device_name(0))

processor = AutoProcessor.from_pretrained(
    model_path,
    local_files_only=True,
    trust_remote_code=False,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_path,
    local_files_only=True,
    trust_remote_code=False,
    torch_dtype="auto",
    device_map="auto",
)

print("Processor:", type(processor).__name__)
print("Modelo:", type(model).__name__)
print("Mapa de dispositivos:", getattr(model, "hf_device_map", None))
print("Carga local completada")

GPU: Tesla T4


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Processor: Qwen2_5_VLProcessor
Modelo: Qwen2_5_VLForConditionalGeneration
Mapa de dispositivos: None
Carga local completada


In [8]:
%cd /content/Resistor-VLM/Resistor-VLM

[Errno 2] No such file or directory: '/content/Resistor-VLM/Resistor-VLM'
/content


In [46]:
!python scripts/run_pipeline.py \
  --image "data/sample_images/resistors/resistor001.jpg" \
  --detector-model "models/detector/best.pt" \
  --vlm-model "/content/models/Qwen2.5-VL-3B-Instruct" \
  --device "cuda" \
  --output-dir "outputs/prototype" \
  --config "configs/default.yaml" \
  --verbose

INFO:resistor_vlm.config.loader:Loaded configuration from configs/default.yaml
DEBUG:pydot:pydot initializing
DEBUG:pydot:pydot 4.0.1
DEBUG:pydot.core:pydot core module initializing
DEBUG:matplotlib:matplotlib data path: /usr/local/lib/python3.12/dist-packages/matplotlib/mpl-data
DEBUG:matplotlib:CONFIGDIR=/root/.config/matplotlib
DEBUG:matplotlib:interactive is False
DEBUG:matplotlib:platform is linux
DEBUG:matplotlib:CACHEDIR=/root/.cache/matplotlib
DEBUG:matplotlib.font_manager:Using fontManager instance from /root/.cache/matplotlib/fontlist-v390.json
INFO:resistor_vlm.detector.backends.model_loader:Loaded model best.pt into cache
INFO:resistor_vlm.detector.backends.yolo_backend:YOLO backend loaded model best.pt on cuda
INFO:resistor_vlm.detector.global_detector:Global detection completed for data/sample_images/resistors/resistor001.jpg: 0 detection(s) in 7703.953 ms
DEBUG:resistor_vlm.detector.tiled.tile_detector:Tile r000_c000 at (0, 0): 0 raw, 0 transformed
DEBUG:resistor_vlm.det

In [47]:
!mkdir -p "/content/drive/MyDrive/Resistor-VLM-assets/models"
!cp -r "/content/models/Qwen2.5-VL-3B-Instruct" \
  "/content/drive/MyDrive/Resistor-VLM-assets/models/"

In [48]:
import gc
import torch

for name in (
    "model",
    "processor",
    "inputs",
    "outputs",
    "generated",
):
    globals().pop(name, None)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print(
    "Memoria asignada:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB",
)
print(
    "Memoria reservada:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB",
)

Memoria asignada: 0.0 GB
Memoria reservada: 0.0 GB


In [49]:
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


In [50]:
!nvidia-smi

Tue Aug  4 17:29:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             34W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [51]:
%cd /content/Resistor-VLM/Resistor-VLM

!python scripts/run_pipeline.py \
  --image "data/sample_images/resistors/resistor001.jpg" \
  --detector-model "models/detector/best.pt" \
  --vlm-model "/content/models/Qwen2.5-VL-3B-Instruct" \
  --device "cuda" \
  --output-dir "outputs/prototype" \
  --config "configs/default.yaml"

/content/Resistor-VLM/Resistor-VLM
INFO:resistor_vlm.config.loader:Loaded configuration from configs/default.yaml
INFO:resistor_vlm.detector.backends.model_loader:Loaded model best.pt into cache
INFO:resistor_vlm.detector.backends.yolo_backend:YOLO backend loaded model best.pt on cuda
INFO:resistor_vlm.detector.global_detector:Global detection completed for data/sample_images/resistors/resistor001.jpg: 0 detection(s) in 8156.361 ms
INFO:resistor_vlm.detector.tiled.tile_detector:Tiled detection completed: 16 tiles and 1 detections
INFO:resistor_vlm.detector.fusion.fusion_engine:Fusion completed: global=0 tiled=1 clusters=0 final=1
INFO:resistor_vlm.detector.pipeline:Detector pipeline completed in global_and_tiled mode with 1 detection(s)
INFO:resistor_vlm.evidence.evidence_builder:Built evidence package 929062f2925d8fd27e7f1ef9 with 1 detection evidences
Loading weights: 100% 824/824 [00:00<00:00, 2421.42it/s]
ERROR:__main__:Pipeline failed: Response is not valid JSON


In [9]:
%cd /content/Resistor-VLM/Resistor-VLM
!git pull origin feature/vlm-pipeline

[Errno 2] No such file or directory: '/content/Resistor-VLM/Resistor-VLM'
/content
fatal: not a git repository (or any of the parent directories): .git


In [10]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
from google.colab import userdata
from pathlib import Path
import io
import requests
import shutil
import zipfile

token = userdata.get("GITHUB_TOKEN")

if not token:
    raise RuntimeError("No se encontró GITHUB_TOKEN en Colab Secrets.")

headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}

archive_url = (
    "https://api.github.com/repos/"
    "jluisro-byte/Resistor-VLM/"
    "zipball/feature/vlm-pipeline"
)

response = requests.get(
    archive_url,
    headers=headers,
    timeout=300,
    allow_redirects=True,
)
response.raise_for_status()

zip_path = Path("/content/resistor-vlm.zip")
zip_path.write_bytes(response.content)

extract_root = Path("/content/github_download")
project_dir = Path("/content/Resistor-VLM")

shutil.rmtree(extract_root, ignore_errors=True)
shutil.rmtree(project_dir, ignore_errors=True)
extract_root.mkdir(parents=True)

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall(extract_root)

folders = [item for item in extract_root.iterdir() if item.is_dir()]

if len(folders) != 1:
    raise RuntimeError(f"Estructura inesperada: {folders}")

shutil.move(str(folders[0]), str(project_dir))
shutil.rmtree(extract_root)

print("Proyecto descargado en:", project_dir)

Proyecto descargado en: /content/Resistor-VLM


In [12]:
from pathlib import Path

matches = list(Path("/content").rglob("pyproject.toml"))

for match in matches:
    print(match)

KeyboardInterrupt: 

In [18]:
%cd /content/Resistor-VLM/Resistor-VLM

/content/Resistor-VLM/Resistor-VLM


In [16]:
%cd /content/Resistor-VLM/Rwsisotoe-VLM

[Errno 2] No such file or directory: '/content/Resistor-VLM/Rwsisotoe-VLM'
/content/Resistor-VLM


In [15]:
from pathlib import Path

for path in [
    Path("pyproject.toml"),
    Path("scripts/run_pipeline.py"),
    Path("configs/default.yaml"),
]:
    print(path, "→", path.exists())

pyproject.toml → False
scripts/run_pipeline.py → False
configs/default.yaml → False


In [17]:
%ls

Resistor-VLM/


In [19]:
from pathlib import Path

for path in [
    Path("pyproject.toml"),
    Path("scripts/run_pipeline.py"),
    Path("configs/default.yaml"),
]:
    print(path, "→", path.exists())

pyproject.toml → True
scripts/run_pipeline.py → True
configs/default.yaml → True


In [20]:
!python -m pip install --upgrade pip
!python -m pip install -e ".[vlm-qwen]"
!python -m pip install ultralytics qwen-vl-utils accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.3 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Obtaining file:///content/Resistor-VLM/Resistor-VLM
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for resistor-vlm (pyproject.toml) ... done
  Created wheel for resistor-vlm: filename=resistor_vlm-1.2.1-0.editable-py3-none-any.whl size=5607 sha256=3777bf95a48e6216b47ee1c9035ebc006565d1a3adbb3a3ddae7fb789932b791
  Stored in directory: /tmp/pip-ephem-wheel-cache-spn8q9yh/wheels/d4/ed/9a/0289066609953a1ea68f8becbe5698a35d3069018739987054
Successfully built resistor-vlm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5

In [21]:
!mkdir -p models/detector
!mkdir -p data/sample_images/resistors
!mkdir -p outputs/prototype

In [22]:
!cp "/content/drive/MyDrive/Resistor-VLM-assets/detector/best.pt" \
    "models/detector/best.pt"

!cp "/content/drive/MyDrive/Resistor-VLM-assets/images/resistor001.jpg" \
    "data/sample_images/resistors/resistor001.jpg"

In [23]:
from pathlib import Path

drive_model = Path(
    "/content/drive/MyDrive/Resistor-VLM-assets/models/"
    "Qwen2.5-VL-3B-Instruct"
)

print("Modelo en Drive:", drive_model.exists())

Modelo en Drive: True


In [24]:
!mkdir -p /content/models
!cp -r \
  "/content/drive/MyDrive/Resistor-VLM-assets/models/Qwen2.5-VL-3B-Instruct" \
  "/content/models/"

In [25]:
import torch
from pathlib import Path

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

checks = [
    Path("models/detector/best.pt"),
    Path("data/sample_images/resistors/resistor001.jpg"),
    Path("/content/models/Qwen2.5-VL-3B-Instruct/config.json"),
]

for path in checks:
    print(path, "→", path.exists())

CUDA: True
GPU: Tesla T4
models/detector/best.pt → True
data/sample_images/resistors/resistor001.jpg → True
/content/models/Qwen2.5-VL-3B-Instruct/config.json → True


In [26]:
!python scripts/run_pipeline.py \
  --image "data/sample_images/resistors/resistor001.jpg" \
  --detector-model "models/detector/best.pt" \
  --vlm-model "/content/models/Qwen2.5-VL-3B-Instruct" \
  --device "cuda" \
  --output-dir "outputs/prototype" \
  --config "configs/default.yaml"

INFO:resistor_vlm.config.loader:Loaded configuration from configs/default.yaml
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
INFO:resistor_vlm.detector.backends.model_loader:Loaded model best.pt into cache
INFO:resistor_vlm.detector.backends.yolo_backend:YOLO backend loaded model best.pt on cuda
INFO:resistor_vlm.detector.global_detector:Global detection completed for data/sample_images/resistors/resistor001.jpg: 0 detection(s) in 10416.287 ms
INFO:resistor_vlm.detector.tiled.tile_detector:Tiled detection completed: 16 tiles and 1 detections
INFO:resistor_vlm.detector.fusion.fusion_engine:Fusion completed: global=0 tiled=1 clusters=0 final=1
INFO:resistor_vlm.detector.pipeline:Detector pipeline completed in global_and_tiled m

In [27]:
%cd /content/Resistor-VLM/Resistor-VLM

/content/Resistor-VLM/Resistor-VLM


In [28]:
!git log -3 --oneline
!grep -n "balanced\|fence\|maximum_new_tokens" resistor_vlm/vlm/response_parser.py resistor_vlm/vlm/backends/qwen_backend.py

fatal: not a git repository (or any of the parent directories): .git
resistor_vlm/vlm/response_parser.py:115:    """Return direct, fenced, or first balanced JSON-object text only."""
resistor_vlm/vlm/backends/qwen_backend.py:37:        maximum_new_tokens: int = 256,
resistor_vlm/vlm/backends/qwen_backend.py:52:        self._tokens = maximum_new_tokens


In [29]:
!git pull origin feature/vlm-pipeline

fatal: not a git repository (or any of the parent directories): .git


In [30]:
!git log -3 --oneline
!grep -n "balanced\|fence\|maximum_new_tokens" resistor_vlm/vlm/response_parser.py resistor_vlm/vlm/backends/qwen_backend.py

fatal: not a git repository (or any of the parent directories): .git
resistor_vlm/vlm/response_parser.py:115:    """Return direct, fenced, or first balanced JSON-object text only."""
resistor_vlm/vlm/backends/qwen_backend.py:37:        maximum_new_tokens: int = 256,
resistor_vlm/vlm/backends/qwen_backend.py:52:        self._tokens = maximum_new_tokens
